## 基本环境 · Basic setup

首次打开运行下面 3 个 cell。它们做的事:
1. 把工作目录切到 `solutions/` (这样 `from attention.mha import ...` 这种导入能直接生效)。
2. 启用 `autoreload`，编辑 .py 文件保存后 notebook 里立刻可用，不用重启 kernel。
3. 设 `LAYERNORM_TYPE=torch`，避免 CUDA-only 算子的 import 失败。

First time you open the notebook, run the 3 cells below: cd into `solutions/`, turn on autoreload, force the pure-PyTorch LayerNorm path.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) != 'solutions':
    if os.path.isdir('solutions'):
        os.chdir('solutions')
    else:
        # already inside a chapter folder — climb out
        while os.path.basename(os.getcwd()) != 'solutions' and os.getcwd() != '/':
            os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
os.environ.setdefault('LAYERNORM_TYPE', 'torch')
print('cwd =', os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
# Folder where the diffusion chapter's reference .pt files live
control_folder = 'diffusion/control_values'
assert os.path.isdir(control_folder), f'missing {control_folder}'

# 第 4 章 · Diffusion

AF3 的结构预测头是一个 [EDM](https://arxiv.org/abs/2206.00364) 风格的扩散模型: 在原子坐标空间逐步去噪。本章实现扩散的三个核心模块, 以及 EDM 采样循环本身 (后者在端到端 notebook 里再用)。

| 文件 | 类 / 函数 | 算法编号 |
|---|---|---|
| `diffusion_transformer.py` | `ConditionedTransitionBlock` | 25 |
| `diffusion_transformer.py` | `DiffusionTransformerBlock` | 23 (单块) |
| `diffusion_transformer.py` | `DiffusionTransformer` | 23 (整 stack) |
| `diffusion_module.py` | `DiffusionConditioning`, `DiffusionModule.f_forward / forward` | 21 / 20 |
| `sampler.py` | `sample_diffusion` | 18 |

## 4.1 ConditionedTransitionBlock (算法 25)

打开 `diffusion/diffusion_transformer.py`，把 `ConditionedTransitionBlock.forward` 的 TODO 填好。

AdaLN 调制 + SwiGLU FFN + adaLN-Zero 输出门 (`linear_s` 的 sigmoid)，是 DiffusionTransformer 每个 block 的 FFN 分支。

In [ ]:
from diffusion.diffusion_transformer import ConditionedTransitionBlock
from diffusion.control_values.diffusion_checks import (
    c_a, c_s, c_z, n_heads, n_blocks, test_inputs,
    test_module_shape, test_module_method,
)

ctb = ConditionedTransitionBlock(c_a=c_a, c_s=c_s, n=2, biasinit=-2.0)
test_module_shape(ctb, 'conditioned_transition_block', control_folder)
test_module_method(
    ctb, 'conditioned_transition_block',
    inputs=(test_inputs['a'], test_inputs['s']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s: ctb(a=a, s=s),
)
print('ConditionedTransitionBlock ✓')

## 4.2 DiffusionTransformerBlock (算法 23 — 单块)

在同一个文件里, 把 `DiffusionTransformerBlock.forward` 填好。

结构: `AttentionPairBias(a, s, z)` (上一章已实现) + DropPath + 残差; 接着`ConditionedTransitionBlock` + DropPath + 残差。注意 `s` / `z` 是原样透传以便激活检查点存活。

In [ ]:
from diffusion.diffusion_transformer import DiffusionTransformerBlock

def _disable_efficient_attn(mod):
    for m in mod.modules():
        if hasattr(m, 'use_efficient_implementation'):
            m.use_efficient_implementation = False

dtb = DiffusionTransformerBlock(c_a=c_a, c_s=c_s, c_z=c_z, n_heads=n_heads)
_disable_efficient_attn(dtb)
test_module_shape(dtb, 'diffusion_transformer_block', control_folder)
test_module_method(
    dtb, 'diffusion_transformer_block',
    inputs=(test_inputs['a'], test_inputs['s'], test_inputs['z']),
    output_names='a_out',
    control_folder=control_folder,
    method=lambda a, s, z: dtb(a=a, s=s, z=z)[0],
)
print('DiffusionTransformerBlock ✓')

## 4.3 DiffusionTransformer (算法 23 — 整 stack)

同一个文件, 把 `DiffusionTransformer.forward` 填好。

把 `n_blocks` 个 DiffusionTransformerBlock 顺序应用; `s` / `z` 在 block 之间不变。

In [ ]:
from diffusion.diffusion_transformer import DiffusionTransformer

dt = DiffusionTransformer(
    c_a=c_a, c_s=c_s, c_z=c_z,
    n_blocks=n_blocks, n_heads=n_heads,
)
_disable_efficient_attn(dt)
test_module_shape(dt, 'diffusion_transformer', control_folder)
test_module_method(
    dt, 'diffusion_transformer',
    inputs=(test_inputs['a'], test_inputs['s'], test_inputs['z']),
    output_names='out',
    control_folder=control_folder,
    method=lambda a, s, z: dt(a=a, s=s, z=z),
)
print('DiffusionTransformer ✓')

## 章节小结

本章实现了 AF3 扩散主干所需的 Transformer 三件套。剩下的两块——DiffusionConditioning (算法 21)、DiffusionModule (算法 20 EDM scaling) 以及顶层 `sample_diffusion` (算法 18)——都已经写好详细的 TODO 伪代码, 学完后续 notebook 后端到端推理就会一次跑通。